To run this notebook, you will need a `.env` file at the root of the project. It should contain the following keys:
```
LLM_SERVICE=OpenAI
OPENAI_API_KEY=...
OPENAI_ENDPOINT=...
OPENAI_DEPLOYMENT_NAME=...
OPENAI_API_VERSION=...
```

The first step is to load the libraries we'll be using. I am importing `pandas` for basic data manipulation, and a package called [discovery_utils](https://github.com/nestauk/discovery_utils) that Karlis made. It contains various functions that we have used and reused across Discovery projects, including functions for extracting structured information using LLMs. You can read more about how the `llm` module works [here](https://github.com/nestauk/discovery_utils/wiki/Checking-data-with-LLM).

In [ ]:
import pandas as pd
from discovery_utils.utils.llm import batch_check

from discovery_heat_pump_futures import PROJECT_DIR

Karlis has created two datasets for this project: one containing research abstracts from [OpenAlex](https://openalex.org/), and one from [Google Patents](https://patents.google.com/).

In [ ]:
openalex_df = pd.read_csv(
    "https://discovery-hub-open-data.s3.eu-west-2.amazonaws.com/future_heat_pumps/heat_pumps_openalex.csv"
)
patents_df = pd.read_json(
    "https://discovery-hub-open-data.s3.eu-west-2.amazonaws.com/future_heat_pumps/heat_pumps_patents.json",
    lines=True,
)

The patents dataset looks like this:

In [ ]:
patents_df.head()

Below, we specify a helper function that can also be used on the OpenAlex data to concatenate the titles and abstracts of individual patents/abstracts. This new `"title_abstract"` field will be the input to `LLMProcessor`.

In [ ]:
def format_title_abstract(
    df: pd.DataFrame, title: str = "title", abstract: str = "abstract"
) -> pd.DataFrame:
    """Format the title and abstract for LLM input.

    Args:
        df (pd.DataFrame): DataFrame containing the title and abstract columns.
        title (str): Name of the title column.
        abstract (str): Name of the abstract column.

    Returns:
        pd.DataFrame: DataFrame with a new column 'title_abstract'
            containing formatted text.
    """
    df["title_abstract"] = (
        "TITLE: "
        + df[title].str.lower().fillna("")
        + " ABSTRACT: "
        + df[abstract].str.lower().fillna("")
    )
    return df


patents_df = format_title_abstract(patents_df, title="title", abstract="abstract")
patents_df.head()

In [ ]:
# to show what the formatted column looks like
patents_df["title_abstract"].values[0]

In [ ]:


# Download OpenAlex dataset
openalex_df = pd.read_csv(
    "https://discovery-hub-open-data.s3.eu-west-2.amazonaws.com/future_heat_pumps/heat_pumps_openalex.csv"
)

# Download Patents dataset
patents_df = pd.read_json(
    "https://discovery-hub-open-data.s3.eu-west-2.amazonaws.com/future_heat_pumps/heat_pumps_patents.json",
    lines=True,
)

# Optional: Save locally for offline reuse
openalex_df.to_csv("heat_pumps_openalex.csv", index=False)
patents_df.to_json("heat_pumps_patents.json", orient="records", lines=True)

print("Datasets downloaded and saved locally.")


In [ ]:
import openai
from openai import OpenAI
client = OpenAI()

In [ ]:
# 0 ── Imports & basic config
import math
from pathlib import Path
from datetime import datetime
import pandas as pd
from tqdm.auto import tqdm
import json
from time import sleep
from dotenv import load_dotenv
import os
from openai import OpenAI
import re

In [ ]:
# -------------------------------
# Load environment and OpenAI client
# -------------------------------
load_dotenv(dotenv_path="/home/pascualdiego/projects/DiscoveryHP/.env")  # Adjust path if needed
client = OpenAI()


In [ ]:
# --------------------------------------------------------------------------- #
# Heat Pump Thesis Classifier  | fully integrated with your environment + KWS #
# --------------------------------------------------------------------------- #
#  What this notebook does end‑to‑end:                                        #
#   1.  Loads patents (JSONL) and research papers (CSV)                       #
#   2.  Calls the OpenAI o3 / gpt‑4o‑mini models to                            #
#         • classify each item into one of 22 sub‑categories                  #
#         • extract short abstract + core innovation                          #
#         • assign cost, efficiency, circularity scores (0‑3)                 #
#NOT YET:#

#   3.  Aggregates the results and produces the Top‑20 lists for each lens    #
#   4.  Saves intermediate files you can explore offline (JSONL, Parquet)     #
#   5.  Lays groundwork for the visualisations (run later)                    #


import os, json, math, pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
from discovery_heat_pump_futures import PROJECT_DIR
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

# Load environment + OpenAI
load_dotenv(dotenv_path=PROJECT_DIR / ".env")
client = OpenAI()

MODEL_NAME = "gpt-4o-mini"
T_CLASSIFY = 0.1
OUTPUT_PATH = PROJECT_DIR / "outputs/fullset_classified_patents.jsonl"
MAX_WORKERS = 10
filter_keywords = True  # ← toggle keyword filtering

# ---- 1. CATEGORIES ---------------------------------------------------------
CATEGORIES = {
    "1.1": "Compressors", "1.2": "Refrigerants", "1.3": "Heat-exchangers", "1.4": "Motor & Drives", "1.5": "Lubrication & oil management",
    "2.1": "Elastocaloric", "2.2": "Electrocaloric", "2.3": "Magnetocaloric", "2.4": "Ionocaloric", "2.5": "Barocaloric",
    "2.6": "Thermoelectric", "2.7": "Electro- & Chemisorption", "2.8": "Thermoacoustic", "2.9": "Sorption / Absorption", "2.10": "Hybrid & cascade",
    "3.1": "Topology & configuration", "3.2": "Controls & optimisation", "3.3": "Operational integration",
    "4.1": "Flexible cycles", "4.2": "Defrost & icing mitigation", "4.3": "Thermal storage",
    "5.1": "Design-for-disassembly", "5.2": "Modular assemblies", "5.3": "Recycled materials", "5.4": "Additive manufacturing", "5.5": "Predictive maintenance"
}

# ---- 2. KEYWORD UNIVERSE ---------------------------------------------------
KWS = {
    "1.1": ["scroll", "rotary", "vane", "twin-screw", "reciprocating", "isothermal compressor", "oil-free", "variable-speed", "economiser"],
    "1.2": ["R32", "R454B", "R290", "propane", "CO₂", "carbon dioxide", "low-GWP", "natural refrigerant", "azeotrope", "zeotropic", "ionic liquid"],
    "1.3": ["micro-channel", "plate heat exchanger", "fin-tube", "enthalpy exchanger", "phase-change heat exchanger", "anti-fouling", "frost-free", "3-D printed"],
    "1.4": ["IPM motor", "SiC inverter", "PMSM", "sensor-less", "flux-weakening"],
    "1.5": ["oil-separator", "mist injection", "CRII", "low-viscosity oil"],
    "2.1": ["elastocaloric", "shape-memory", "Ni-Ti"], "2.2": ["electrocaloric", "ferroelectric"], "2.3": ["magnetocaloric", "gadolinium"],
    "2.4": ["ionocaloric", "ion-solvation"], "2.5": ["barocaloric"], "2.6": ["thermoelectric", "Peltier", "Seebeck"],
    "2.7": ["electrochemical compressor", "chemisorption"], "2.8": ["thermoacoustic"],
    "2.9": ["adsorption heat pump", "absorption heat pump", "lithium bromide"], "2.10": ["cascade heat pump", "hybrid heat pump"],
    "3.1": ["cascade", "booster", "trans-critical", "bi-valent", "ground-source"], "3.2": ["model predictive control", "MPC", "PID", "fault detection", "digital twin"],
    "3.3": ["smart-grid", "thermal district network", "hybrid boiler"], "4.1": ["ejector cycle", "regenerative cycle", "parallel-compressor"],
    "4.2": ["defrost", "icing sensor", "hot-gas bypass", "nano-coating"], "4.3": ["PCM storage", "phase change material", "heat battery", "stratified tank"],
    "5.1": ["tool-less", "snap-fit", "reversible adhesive", "design-for-disassembly", "DfD"], "5.2": ["modular cartridge", "remanufactur", "refurbish", "life-extension"],
    "5.3": ["recycled", "bio-polymer", "PCR plastic", "reclaimed copper", "low-GWP foam"], "5.4": ["additive manufacturing", "3-D print", "near-net shape", "binder-jet"],
    "5.5": ["predictive maintenance", "condition monitoring", "service-as-a-product"]
}
KW2CAT = {kw.lower(): cat for cat, kwlist in KWS.items() for kw in kwlist}

# ---- 3. ISO HEURISTIC ------------------------------------------------------
ISO_LEVERS = {
    "maintain":  ["predictive maintenance", "condition monitoring", "service-as-a-product"],
    "reuse":     ["remanufactur", "refurbish", "cartridge", "core exchange"],
    "recycle":   ["recycled", "reclaim", "mono-material"],
    "reduce":    ["near-net shape", "additive manufacturing", "yield >"],
    "regenerate":["bio-based", "bio-polymer", "renewable feedstock"]
}
def iso_score(text):
    hits = []
    low = text.lower()
    for kws in ISO_LEVERS.values():
        n = sum(1 for kw in kws if kw in low)
        hits.append(0 if n==0 else 1 if n==1 else 2 if n==2 else 3)
    return math.ceil(sum(hits) / 5)

# ---- 4. PROMPT GENERATOR ---------------------------------------------------
def make_prompt(title, abstract):
    return f"""
You are a sustainable heating technology analyst.

TITLE: {title}
ABSTRACT: {abstract}

Answer:
1. What is the core technical innovation? (max 25 words)
2. Is it relevant to heat pumps? ('yes' or 'no')
3. If yes, assign exactly ONE category code: {', '.join(CATEGORIES.keys())}
4. Score 0-3 for:
   a) cost_reduction_potential
   b) efficiency_gain_potential
   c) circularity_score (use ISO 59004 lever heuristic)
Respond ONLY as JSON:
{{"is_relevant": "...", "summary": "...", "category": "...",
  "cost_score": ..., "efficiency_score": ..., "circularity_score": ...}}
""".strip()

# ---- 5. RUN PARALLEL CLASSIFICATION ----------------------------------------
print("▶︎ Loading and preparing data ...")
patents_df = pd.read_json("heat_pumps_patents.json", lines=True)
print("Total patents loaded:", len(patents_df))

if filter_keywords:
    patents_df["text"] = (patents_df["title"] + " " + patents_df["abstract"]).str.lower()
    patents_df = patents_df[patents_df["text"].apply(lambda t: any(k in t for k in KW2CAT))].copy()
    print("After keyword filtering:", len(patents_df))

# Load already processed IDs
processed_ids = set()
if OUTPUT_PATH.exists():
    with open(OUTPUT_PATH, "r") as f:
        processed_ids = {json.loads(line)["publication_number"] for line in f}
    print("Resuming from checkpoint. Already done:", len(processed_ids))

# Classifier function
def classify(row):
    if row["publication_number"] in processed_ids:
        return None
    try:
        prompt = make_prompt(row["title"], row["abstract"])
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=T_CLASSIFY,
            response_format={"type": "json_object"}
        )
        result = json.loads(response.choices[0].message.content)
        result.update({
            "publication_number": row.get("publication_number", ""),
            "title": row["title"],
            "abstract": row["abstract"],
            "url": row.get("url", "")
        })
        return result
    except Exception as e:
        print(f"⚠️ Error on {row.get('publication_number', '')}: {e}")
        return None

# Run it
print("▶︎ Starting classification ...")
t0 = datetime.now()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor, open(OUTPUT_PATH, "a") as f_out:
    futures = [executor.submit(classify, row) for _, row in patents_df.iterrows()]
    for future in tqdm(as_completed(futures), total=len(futures)):
        result = future.result()
        if result:
            f_out.write(json.dumps(result) + "\n")
print(f"🎉 Done in {datetime.now() - t0}")
print(f"✅ Output saved to: {OUTPUT_PATH}")

In [ ]:
# --------------------------------------------------------------------------- #
# 1 and 2. CLASSIFY OPENALEX RESEARCH PAPERS (Same pipeline as patents)             #
# --------------------------------------------------------------------------- #

OPENALEX_PATH = "heat_pumps_openalex.csv"
OPENALEX_OUTPUT = PROJECT_DIR / "outputs/fullset_classified_papers.jsonl"

print("▶︎ Loading OpenAlex research dataset ...")
openalex_df = pd.read_csv(OPENALEX_PATH)
print("Total papers loaded:", len(openalex_df))

# Optional: filter rows by keyword if enabled
if filter_keywords:
    openalex_df["text"] = (
        openalex_df["title"].fillna('') + " " + openalex_df["abstract"].fillna('')
    ).str.lower()

    openalex_df = openalex_df[openalex_df["text"].str.strip() != ""]

    openalex_df = openalex_df[
        openalex_df["text"].apply(lambda t: any(k in t for k in KW2CAT))
    ].copy()

    print("After keyword filtering:", len(openalex_df))

# Load already processed DOIs
processed_dois = set()
if OPENALEX_OUTPUT.exists():
    with open(OPENALEX_OUTPUT, "r") as f:
        processed_dois = {json.loads(line).get("doi_url", "") for line in f}
    print("Resuming from checkpoint. Already done:", len(processed_dois))

# Build prompt for research paper
def make_prompt_paper(title, abstract):
    return f"""
You are a sustainable heating technology analyst.

TITLE: {title}
ABSTRACT: {abstract}

Answer:
1. What is the core technical innovation? (max 25 words)
2. Is it relevant to heat pumps? ('yes' or 'no')
3. If yes, assign exactly ONE category code: {', '.join(CATEGORIES.keys())}
4. Score 0-3 for:
   a) cost_reduction_potential
   b) efficiency_gain_potential
   c) circularity_score (use ISO 59004 lever heuristic)
Respond ONLY as JSON:
{{"is_relevant": "...", "summary": "...", "category": "...",
  "cost_score": ..., "efficiency_score": ..., "circularity_score": ...}}
""".strip()

# LLM wrapper
def classify_paper(row):
    doi = row.get("doi_url", "")
    if doi in processed_dois:
        return None
    try:
        prompt = make_prompt_paper(row["title"], row["abstract"])
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=T_CLASSIFY,
            response_format={"type": "json_object"}
        )
        result = json.loads(response.choices[0].message.content)
        result.update({
            "title": row["title"],
            "abstract": row["abstract"],
            "doi_url": doi
        })
        return result
    except Exception as e:
        print(f"⚠️ Error on paper: {e}")
        return None

# Run in parallel
print("▶︎ Starting OpenAlex classification ...")
t0 = datetime.now()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor, open(OPENALEX_OUTPUT, "a") as f_out:
    futures = [executor.submit(classify_paper, row) for _, row in openalex_df.iterrows()]
    for future in tqdm(as_completed(futures), total=len(futures)):
        result = future.result()
        if result:
            f_out.write(json.dumps(result) + "\n")
print(f"🎉 Done in {datetime.now() - t0}")
print(f"✅ Output saved to: {OPENALEX_OUTPUT}")
